In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path(
    "/Users/glebsavelev/Desktop/da/diabetes_prediction_dataset.csv"
)

df = pd.read_csv(DATA_PATH)

print("Розмір датасету:", df.shape)
print("Кількість дублікатів:", df.duplicated().sum())

display(df.head())

Розмір датасету: (100000, 9)
Кількість дублікатів: 3854


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


Крок 2. Видалення повних дублікатів
Навіщо це потрібно?
У нашій таблиці є повністю однакові рядки. Якщо однакові записи потраплять одночасно до навчальної та тестової вибірок, це може призвести до завищеної оцінки якості моделі.
Водночас однакові медичні показники не обов'язково означають, що записи належать одній людині. Тому видалення дублікатів — це обране нами методологічне рішення, а не доказ помилки вихідного набору даних.
Для основного експерименту використаємо набір без повних дублікатів.

In [2]:
# Створюємо копію без повних дублікатів
df_clean = df.drop_duplicates().reset_index(drop=True)

print("Початкова кількість записів:", len(df))
print("Кількість видалених дублікатів:", len(df) - len(df_clean))
print("Кількість записів після очищення:", len(df_clean))

print("\nРозподіл класів після очищення:")
print(df_clean["diabetes"].value_counts())

print("\nЧастка класів (%):")
print(
    df_clean["diabetes"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Початкова кількість записів: 100000
Кількість видалених дублікатів: 3854
Кількість записів після очищення: 96146

Розподіл класів після очищення:
diabetes
0    87664
1     8482
Name: count, dtype: int64

Частка класів (%):
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64


Крок 3 — розділення даних на навчальну та тестову вибірки
Тепер нам потрібно розділити 96 146 записів:
- 80% — навчальна вибірка. На ній будемо навчати моделі та підбирати їхні параметри через кросвалідацію.
- 20% — тестова вибірка. Використаємо її лише для остаточного оцінювання моделей.
Для цього застосуємо train_test_split із параметром stratify=y, який допомагає зберегти пропорції класів в обох вибірках.

In [4]:
from sklearn.model_selection import train_test_split

# Відокремлюємо ознаки від цільової змінної
X = df_clean.drop(columns=["diabetes"])
y = df_clean["diabetes"]

# Розподіляємо дані зі збереженням пропорцій класів
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Навчальна вибірка:", X_train.shape)
print("Тестова вибірка:", X_test.shape)

print("\nРозподіл класів у навчальній вибірці (%):")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nРозподіл класів у тестовій вибірці (%):")
print(y_test.value_counts(normalize=True).mul(100).round(2))

# Переконуємося, що індекси не перетинаються
assert set(X_train.index).isdisjoint(set(X_test.index))

print("\nПеревірка успішна: навчальні та тестові індекси не перетинаються.")

Навчальна вибірка: (76916, 8)
Тестова вибірка: (19230, 8)

Розподіл класів у навчальній вибірці (%):
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64

Розподіл класів у тестовій вибірці (%):
diabetes
0    91.18
1     8.82
Name: proportion, dtype: float64

Перевірка успішна: навчальні та тестові індекси не перетинаються.


In [ ]:
Крок 4 — підготовка ознак через Pipeline
Наш датасет містить два види ознак:
- Категоріальні: gender, smoking_history.
- Числові: age, hypertension, heart_disease, bmi, HbA1c_level, blood_glucose_level.
Алгоритмам машинного навчання потрібне числове представлення даних. Тому категорії перетворимо через OneHotEncoder, а числові показники масштабуємо за допомогою StandardScaler.
Використаємо ColumnTransformer, щоб об'єднати ці операції.

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

categorical_features = ["gender", "smoking_history"]

numeric_features = [
    "age",
    "hypertension",
    "heart_disease",
    "bmi",
    "HbA1c_level",
    "blood_glucose_level"
]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

print("Попередню обробку ознак налаштовано.")
print("Числових ознак:", len(numeric_features))
print("Категоріальних ознак:", len(categorical_features))

Попередню обробку ознак налаштовано.
Числових ознак: 6
Категоріальних ознак: 2


In [ ]:
Крок 5. Збереження розподілу даних
Зараз завершимо роботу у файлі 02_preprocessing.ipynb.
Щоб у наступному ноутбуці 03_model_training.ipynb 
використовувати ті самі навчальну й тестову вибірки, збережемо їх у папці results/data.

In [6]:
from pathlib import Path

# Папка для збереження підготовлених даних
OUTPUT_DIR = Path.cwd() / "results" / "data"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Зберігаємо ознаки та цільові змінні окремо
X_train.to_csv(OUTPUT_DIR / "X_train.csv", index=False)
X_test.to_csv(OUTPUT_DIR / "X_test.csv", index=False)

y_train.to_csv(OUTPUT_DIR / "y_train.csv", index=False)
y_test.to_csv(OUTPUT_DIR / "y_test.csv", index=False)

print("Дані успішно збережено!")
print("Папка:", OUTPUT_DIR)

print("\nЗбережені файли:")
for file in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", file.name)

Дані успішно збережено!
Папка: /Users/glebsavelev/Desktop/da/results/data

Збережені файли:
- X_test.csv
- X_train.csv
- y_test.csv
- y_train.csv
